In [ ]:
#%%apyter init
from appyter import magic
magic.init(lambda _=globals: _())

In [ ]:
%%appyter hide_code

{% do SectionField(
    name='input', 
    title = '1. Gene Upload', 
    subtitle = 'Enter a single human gene or a gene set'
) %}

{% do SectionField(
    name='analysis', 
    title = '2. Select a drug ranking method for single gene analysis', 
    subtitle = "Select a ranking method by which to identify top up- and down-regulating drugs. Options are to rank drugs by the differential expression (DE) p-value of the target (default) or by the rank of the target in each perturbation's DE signature (Target Rank), where ranking is determined by the adjusted p-value of differential expression. This method prioritizes drug specificity over magnitude of regulation. THIS PARAMETER IS IGNORED for the multi-gene analysis."
) %}

In [ ]:
%%appyter hide code
{% set gene_input = TabField(
    name = 'gene_input',
    label = 'Query Gene(s)',
    default = 'Single Gene',
    description = 'Enter the gene symbol or gene set of interest.',
    choices = {
        'Single Gene': [
            AutocompleteField(
                name='single_gene_input',
                label = 'Input Gene',
                default = 'C9ORF72',
                description = 'Enter a single human gene of interest',
                file_path = 'https://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/all_genes.json',
                section = 'input'
            )
        ],
        'Multi-gene': [
            TextField(
                name='multi_gene_input',
                label = 'Input Gene Set',
                default = 'GPR141\nMS4A6A\nCR1\nHLA-DQA1\nMS4A4A',
                description = 'Enter a list of genes, one per row',
                section = 'input'
            )
        ]
    },
    section='input'
)%}

In [ ]:
%%appyter hide_code
{% set ranking_method = ChoiceField(
    name = 'ranking_method',
    label = 'Ranking Method',
    default = 'Differential Expression P-Value',
    description = "Rank drugs by the differential expression (DE) p-value of the target or the rank of the target in each perturbation's DE signature. Ranking is determined by the adjusted p-value of differential expression.",
    choices = [
        'Differential Expression P-Value',
        'Target Rank'
    ],
    section='analysis'
)%}

In [ ]:
%%appyter code_exec

{%- if gene_input.raw_value == 'Single Gene' %} # single gene input
input_type = 'single'
query_gene = "{{ gene_input.value[0]['args']['value'].upper() }}"
{%- else -%} # multi-gene input
geneset = {{ gene_input.value[0] }}
input_type = 'multi'
{%- endif %}

{%- if ranking_method.raw_value == 'Differential Expression P-Value' %}
ranking_method = 'pval'
{%- else %}
ranking_method = 'target_rank'
{%- endif %}

In [ ]:
%%appyter code_exec
# this cell parses the gene list into a python list

{%- if gene_input.raw_value == 'Multi-gene' %}
query_geneset = geneset.split('\n')
query_geneset = [x.strip().upper() for x in query_geneset]
{%- endif %}

# DrugRanger

This notebook takes a gene as input and identifies drugs that maximally up and down regulate the gene's mRNA expression in a collection of connectivity mapping resources that measure transcriptional response to chemical perturbations:

- Ginkgo GDPx1 and GPDx2: Limma-Voom based differential gene expression results for 1,354 drugs.
- Novartis DRUG-seq: Differential: Limma-Trend based differential gene expression results for 4,343 drugs. 
- Tahoe 100-M: DESeq based differential gene expression results for 376 drugs tested across 50 different cancer cell lines. 

The Ginkgo dataset includes 4 primary cell types (epithelial melanocytes, smooth aortic muscle cells, skeletal muscle myoblasts and dermal fibroblasts) and one cell line (A549 lung carcinoma cell line). Previous analysis showed distinct transcriptional responses by cell type, so the drug rankings for the Ginkgo dataset are separated by cell type.

The Deepcover MoA proteomics dataset is used to present protein-level regulation of the query gene. You can compare protein-level and mRNA-level regulation for compounds used in both the Deepcover MoA and connectivity mapping resources.

In [ ]:
## General
import pandas as pd
import numpy as np
import re
from itertools import combinations
import warnings
import hashlib

## Helpers
from cmap_readers import *
from multi_gene_utils import *

## Tables
from IPython.display import display, display_markdown, HTML, FileLink

## UpSet Plot
from upsetplot import from_contents, plot

## Venn Diagram
from matplotlib_venn import venn3, venn2
import matplotlib.pyplot as plt

## Volcano Plot
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool, LinearColorMapper, CategoricalColorMapper
from bokeh.palettes import RdBu, Category10
from bokeh.io import output_notebook

## Barplots
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colorbar import ColorbarBase
import matplotlib.patches as mpatches

In [ ]:
# Storage URLs for DE gene files
ginkgo_URL = 'https://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/ginkgo_de'
novartis_URL = 'https://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/novartis_de'
deepcover_moa_URL = 'https://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/deepcoverMoa_de'
tahoe_URL = 'https://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/tahoe_de'

# silence warnings
warnings.filterwarnings('ignore')

In [ ]:
in_ginkgo = in_novartis = in_tahoe = True
ginkgo_cell_types = ['human_epithelial_melanocytes', 'human_dermal_fibroblast', 'human_aortic_smooth_muscle_cells', 'human_skeletal_muscle_myoblasts','A549']

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
# get Ginkgo DE results for gene
ginkgo_gene_expr_dict = prepare_ginkgo_data_dict(query_gene, ginkgo_cell_types, ginkgo_URL)
if ginkgo_gene_expr_dict == None:
    in_ginkgo=False
    print('Gene not in Ginkgo dataset')   
# get Novartis DE results for gene
novartis_de = prepare_novartis_data(query_gene, novartis_URL)
if novartis_de is None:
    print('Gene not in Novartis DRUG-seq dataset')
    in_novartis = False
# get Tahoe DE results for gene
def hash_bucket(gene, num_buckets=512):
    '''
    gene: Gene symbol
    num_buckets: number of hash buckets to create

    Returns integer hash for gene name (between 0-n_buckets)
    '''
    return int(hashlib.md5(gene.encode()).hexdigest(),16) % num_buckets
query_gene_encoded = hash_bucket(query_gene)
tahoe_de = pd.read_parquet(f'{tahoe_URL}/gene_bucket_{query_gene_encoded}.parquet')
tahoe_de = prepare_tahoe_data(tahoe_de, query_gene)
if tahoe_de is None:
    print('Gene not in Tahoe-100M dataset')
    in_tahoe=False

if in_novartis + in_ginkgo + in_tahoe < 1:
    print(f"Novartis: {in_novartis}")
    print(f"Ginkgo: {in_ginkgo}")
    print(f"Tahoe-100M: {in_tahoe}")
    raise Exception("Execution stopped, gene not found in any datasets")
# Get proteomics data
in_deepcover = True
try:
    protein_de = pd.read_feather(f'{deepcover_moa_URL}/{query_gene}.f').set_index('index')
except:
    in_deepcover=False
    
{% endif %} 

In [ ]:
# Get pubchem ID dataframe
pubchem_location = 'https://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/cmap_pubchem_ids_10062025.csv'
pubchem_ids = pd.read_csv(pubchem_location, dtype = {'Drug':str, 'CID':str})

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Multi-gene' %}
# Retrieve data for multi-gene analysis
pbc_df = pubchem_ids.copy()
pbc_df.rename(columns={'Drug':'DrugMatch'}, inplace=True)
all_genes, not_found = fetch_cmap_data(query_geneset)
single_gene_drug_ranks = single_gene_rank(all_genes, query_geneset, pbc_df)
combined_gene_results, multi_gene_drug_ranks = get_multi_gene_ranks(single_gene_drug_ranks, pbc_df)

# genes not found in any dataset
if len(not_found) == 3:
    none_found = set.intersection(*[set(gl) for gl in not_found.values()])
    if none_found:
        display_markdown(f'Gene(s) {none_found} not found in Novartis DRUG-seq, Tahoe-100M, or Ginkgo. **Excluded from multi-gene analysis.**', raw=True)
else:
    none_found = {}
    
# list genes not found in some datasets
display_markdown('**Genes not found in the following datasets:**', raw=True)
for ds,gl in not_found.items():
    print(f'{ds}: {gl}')    
if (len(none_found) == len(query_geneset)):
        print('No input genes found in any datasets')
        raise Exception("Execution stopped, no input genes found in any datasets")
{% endif %}

## Query Gene(s)

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
    display_markdown("**Single Gene Input**", raw=True)
    display_markdown(f"This notebook shows results for the input gene **{query_gene}**", raw=True)
    display_markdown(f"Drugs that up and down regulate **{query_gene}** are ranked by method **{ranking_method}**", raw=True)
{% elif gene_input.raw_value == 'Multi-gene' %}
    display_markdown("**Multi-gene Input**", raw=True)
    display_markdown(f"This notebook shows results for the input gene set **{query_geneset}**", raw=True)
{% endif %}

## Drug Tables

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Single Gene' %}

Within each dataset drugs are ranked by either:

1. The statistical significance of the regulatory relationship. The pipeline uses the adjusted p-value from the differential expression results.

or

2. The rank of the query gene relative to all other targets. Rank is determined by differential expression adjusted p-value. A ranking of 1 indicates that the gene has the most significant adjusted p-value for differential expression compared to all other genes for that perturbation. 

When a dataset contains multiple perturbations for the same drug (i.e. a cell exposed to the drug at different doses), p-values and normalized ranks are averaged across doses to get a single ranking for the drug. 

The rankings are done separately for up-regulated and down-regulated genes.

{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
def get_rankings(data:pd.DataFrame, source:str, cell_type:str, direction:str, option:str):
    '''
    Given a dataframe of logFC and p-values for a gene of interest across perturbations, 
    rank the drugs by how the induce or repress the gene. 

    Returns a tuple of 1) drug ranks averaged across drug dosages and 2) full
    perturbation ranks. 
    '''
    ranked_data = data.copy()

    ranked_data['DrugLower'] = ranked_data['Drug'].str.lower()
    ranked_data = ranked_data.merge(pubchem_ids, how='left', left_on ='DrugLower', right_on='Drug')
    ranked_data['DrugGroup'] = ranked_data['CID']
    ranked_data['DrugGroup'] = ranked_data['DrugGroup'].fillna(ranked_data['DrugLower'])
    
    # if (source == 'Ginkgo') & (cell_type != 'A549'):
    #     ranked_data.loc[ranked_data['Drug']=='Brefeldin-A', 'Drug'] = 'Brefeldin A'
    if source == 'Novartis':
        # ranked_data.loc[ranked_data['Drug']=='Trichostatin A (racemate)', 'Drug'] = 'Trichostatin A'
        ranked_data.rename(columns={'P.Adj':'adj.P.Val'}, inplace=True)
    if option == 'pval':
        # average rank across all drug dosages
        drug_mean_ranks = ranked_data.loc[:,['DrugGroup','logFC','log10adj.P.Val']].groupby('DrugGroup')[['logFC','log10adj.P.Val']].mean().sort_values('log10adj.P.Val', ascending=False)
        # filter for up or down regulation
        if direction == 'up':
            drug_mean_ranks = drug_mean_ranks[drug_mean_ranks['logFC'] > 0]
        elif direction == 'down':
            drug_mean_ranks = drug_mean_ranks[drug_mean_ranks['logFC'] < 0]
    elif option == 'target_rank':
        if direction == 'up':
            ranked_data = ranked_data[ranked_data['GeneDir'] == 'Up']
        elif direction == 'down':
            ranked_data = ranked_data[ranked_data['GeneDir'] == 'Dn']
        drug_mean_ranks = ranked_data.loc[:,['DrugGroup','logFC','adj.P.Val','Rank','PctRank']].groupby('DrugGroup')[['logFC','adj.P.Val','Rank','PctRank']].mean().sort_values('PctRank', ascending=True)
    # clean drug mean ranks
    drug_mean_ranks.reset_index(drop=False, names='DrugGroup', inplace=True)
    drug_mean_ranks = drug_mean_ranks.merge(pubchem_ids, how='left', left_on='DrugGroup', right_on='CID')
    drug_mean_ranks['Drug'].fillna(drug_mean_ranks['DrugGroup'], inplace=True)
    if option == 'pval':
        drug_mean_ranks = drug_mean_ranks.drop_duplicates(subset=['CID','logFC','log10adj.P.Val'])
        drug_mean_ranks.rename(columns={'logFC':'Avg logFC', 'log10adj.P.Val':'Avg -log10(Adj.PVal)'}, inplace=True)
    elif option == 'target_rank':
        drug_mean_ranks = drug_mean_ranks.drop_duplicates(subset=['CID','logFC','PctRank'])
        drug_mean_ranks.rename(columns={'logFC':'Avg logFC', 'PctRank':'Avg PctRank'}, inplace=True)
    # clean ranks
    ranked_data = ranked_data[['Gene','Perturbation','Drug_x','CID','DrugGroup','adj.P.Val','logFC','GeneDir','Rank','PctRank','log10adj.P.Val']]
    ranked_data.rename(columns={'Drug_x':'Drug'}, inplace=True)
    return drug_mean_ranks, ranked_data

def get_top(rank_results:pd.DataFrame, n=50):
    '''
    Given the drug_mean_ranks result from get_rankings, extract the names of the drugs
    that most down- or up-regulate the gene of interest (top N).

    If there are less drugs than N, will return all results.
    '''
    top = {d.casefold() for d in set(rank_results.head(n)['Drug'])}
    return top

{% endif %}

def download_link(df, fname, link_header='Download full results'):
    '''
    create download link for table results
    '''
    if df.shape[0] == 0: return ''
    csv = df.to_csv(fname, sep='\t', index=True)
    link = f'<div>{link_header}: <a href="{fname}" target=_blank>{fname}</a></div>'
    return link

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Single Gene' %}
### Ginkgo
Drug rankings for the Ginkgo dataset. Top 20 by the chosen ranking method are shown, and the full results are available for download. 
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
top_n = 20
if in_ginkgo:
    ginkgo_drugs_up = {}
    ginkgo_drugs_down = {}
    for cell_type, exprdf in ginkgo_gene_expr_dict.items():
        # rank by level of up-regulation
        mean_ranks, full_ranks = get_rankings(exprdf, 'Ginkgo', cell_type, 'up', ranking_method)
        ginkgo_drugs_up[cell_type] = (mean_ranks, full_ranks)
        display_markdown(f'**Top {top_n} up-regulators for {cell_type}**', raw=True)
        display(mean_ranks.head(top_n))
        display(HTML(download_link(mean_ranks, f"ginkgo_drug_ranks_{query_gene}_UpReg_{cell_type}.tsv", 'Download results averaged across drug dosages')))
        display(HTML(download_link(full_ranks, f"ginkgo_drug_ranks_{query_gene}_full_UpRg_{cell_type}.tsv", 'Download results for all perturbations')))
        # rank by level of down-regulation
        mean_ranks, full_ranks = get_rankings(exprdf, 'Ginkgo', cell_type, 'down', ranking_method)
        ginkgo_drugs_down[cell_type] = (mean_ranks, full_ranks)
        display_markdown(f'**Top {top_n} down-regulators for {cell_type}**', raw=True)
        display(mean_ranks.head(top_n))
        display(HTML(download_link(mean_ranks, f"ginkgo_drug_ranks_{query_gene}_DnReg_{cell_type}.tsv", 'Download results averaged across drug dosages')))
        display(HTML(download_link(full_ranks, f"ginkgo_drug_ranks_{query_gene}_full_DnRg_{cell_type}.tsv", 'Download results for all perturbations')))
else:
    display_markdown(f'**{query_gene}** not found in Ginkgo datasets', raw=True)

{% endif %}     

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Single Gene' %}
### Novartis DRUG-seq
Drug rankings for the Novartis DRUG-seq dataset. Top 20 by the chosen ranking method are shown, and the full results are available for download. 
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
if in_novartis:
    novartis_drugs_up = get_rankings(novartis_de, 'Novartis', '', 'up', ranking_method)
    novartis_drugs_down = get_rankings(novartis_de, 'Novartis', '', 'down', ranking_method)

    display_markdown(f'**Top {top_n} up-regulators in Novartis DRUG-seq**', raw=True)
    display(novartis_drugs_up[0].head(top_n))
    display(HTML(download_link(novartis_drugs_up[0], f'novartis_drug_ranks_{query_gene}_UpReg.tsv', 'Download results averaged across drug dosages')))
    display(HTML(download_link(novartis_drugs_up[1], f'novartis_drug_ranks_{query_gene}_full_UpReg.tsv', 'Download results for all perturbations')))
    display_markdown(f'**Top {top_n} down-regulators in Novartis DRUG-seq**', raw=True)
    display(novartis_drugs_down[0].head(top_n))
    display(HTML(download_link(novartis_drugs_down[0], f'novartis_drug_ranks_{query_gene}_DnReg.tsv', 'Download results averaged across drug dosages')))
    display(HTML(download_link(novartis_drugs_down[1], f'novartis_drug_ranks_{query_gene}_full_DnReg.tsv', 'Download results for all perturbations')))
else:
    display_markdown(f'**{query_gene}** not found in Novartis DRUG-seq', raw=True)
{% endif %}

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Single Gene' %}
### Tahoe-100M
Drug rankings for the Tahoe-100M dataset. Top 20 by the chosen ranking method are shown, and the full results are available for download. 
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
if in_tahoe:
    tahoe_drugs_up = get_rankings(tahoe_de, 'Tahoe', '', 'up', ranking_method)
    tahoe_drugs_down = get_rankings(tahoe_de, 'Tahoe', '', 'down', ranking_method)

    display_markdown(f'**Top {top_n} up-regulators in Tahoe-100M**', raw=True)
    display(tahoe_drugs_up[0].head(top_n))
    display(HTML(download_link(tahoe_drugs_up[0], f'tahoe_drug_ranks_{query_gene}_UpReg.tsv', 'Download results averaged across drug dosages')))
    display(HTML(download_link(tahoe_drugs_up[1], f'tahoe_drug_ranks_{query_gene}_full_UpReg.tsv', 'Download results for all perturbations')))
    display_markdown(f'**Top {top_n} down-regulators in Tahoe-100M**', raw=True)
    display(tahoe_drugs_down[0].head(top_n))
    display(HTML(download_link(tahoe_drugs_down[0], f'tahoe_drug_ranks_{query_gene}_DnReg.tsv', 'Download results averaged across drug dosages')))
    display(HTML(download_link(tahoe_drugs_down[1], f'tahoe_drug_ranks_{query_gene}_full_DnReg.tsv', 'Download results for all perturbations')))
else:
    display_markdown(f'**{query_gene}** not found in Tahoe-100M', raw=True)
{% endif %}

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Multi-gene' %}

Drug ranks according to the adjusted p-value method are generated for each input gene. A single multi-gene drug rank for each drug is calculated by averaging the ranks for a drug across all input genes. 

Tables below show the top 20 drugs identified by their 0-1 normalized multi-gene drug rank, where lower values correspond to greater regulation, and their multi-gene rank deviation from the expected drug rank. The expected rank for each drug was calculated from random input gene sets. 
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Multi-gene' %}
# expected rank information
null_ranks_dn = pd.read_csv('https://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/null_rank_dn.csv').rename(columns={'MultiGeneRank':'NullRank'})
null_ranks_up = pd.read_csv('https://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/null_rank_up.csv').rename(columns={'MultiGeneRank':'NullRank'})
# Universal regulators
flagged_drugs = pd.read_csv('https://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/flagged_drug_list.csv')
flagged_drugs['DrugMatch'].fillna(flagged_drugs['CID'], inplace=True)
# Data on FDA approved drugs from OpenTargets
fda_approved_toxicity = pd.read_csv('https://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/opentargets_fda_approved_toxicity.csv')
fda_approved_toxicity['synonyms_lower'] = fda_approved_toxicity['synonyms'].str.lower()
fda_approved_toxicity['name_lower'] = fda_approved_toxicity['name'].str.lower()
fda_approved_toxicity.synonyms_lower.fillna(fda_approved_toxicity['name_lower'], inplace=True)
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Multi-gene' %}
def clean_ranks_df(direction, method):
    df = multi_gene_drug_ranks[direction]
    df = df.merge(pbc_df, how='left', on='DrugMatch')
    df = df.drop_duplicates(subset=['DrugGroup', 'MultiGeneRank','NGenes'])
    df['DrugMatch'] = df['DrugMatch'].fillna(df['DrugGroup'])
    # expected ranks
    df = df.merge(null_ranks_up[['DrugMatch','NullRank']], how='left', on='DrugMatch')
    # FDA status
    df = df.merge(fda_approved_toxicity[['synonyms_lower','isApproved']], how='left', left_on='DrugMatch', right_on='synonyms_lower')
    df['isApproved'].fillna(False, inplace=True)

    if method=='mg_rank':
        df = df.sort_values(['MultiGeneRank', 'NGenes'], ascending=[True, False])
        new_names = {
            'DrugMatch':'Drug',
            'CID': 'PubChem CID',
            'MultiGeneRank': f'Multi-gene {direction} Rank',
            'NGenes': 'N Genes Regulated',
            'isApproved':'FDA Approved'
        }
    elif method=='deviation':
        df['RankDiff'] = df['NullRank'] - df['MultiGeneRank']
        df = df.sort_values(['RankDiff','MultiGeneRank', 'NGenes'], ascending=[False, True, False])
        new_names = {
            'DrugMatch':'Drug',
            'CID': 'PubChem CID',
            'MultiGeneRank': f'Multi-gene {direction} Rank',
            'RankDiff': f'Expected Rank Difference {direction}',
            'NGenes': 'N Genes Regulated',
            'isApproved':'FDA Approved'
        }

    df = df[list(new_names.keys())]
    df.rename(columns=new_names, inplace=True)
    df = df.drop_duplicates(subset=['PubChem CID',f'Multi-gene {direction} Rank', 'N Genes Regulated'])

    return df

def clean_individual_df(direction):
    df = combined_gene_results[direction]
    df = df.merge(pbc_df, how='left', on='DrugMatch')
    df = df.drop_duplicates(subset = ['gene','CID','Drug','MeanRank'])
    new_names = {
        'gene':'Gene',
        'Drug':'Drug',
        'CID':'PubChem CID',
        'MeanRank':f'Mean Rank {direction}'
    }
    df = df[list(new_names.keys())]
    df.rename(columns=new_names, inplace=True)
    return df
def filter_genes(df, cutoff):
    return df[df['N Genes Regulated'] > cutoff]
top_n = 20
## multi-gene rank method
# Up Regulation
display_markdown('**Multi-gene Drug Rank Up**', raw=True)
mg_ranks_up = clean_ranks_df('Up', 'mg_rank')
display(filter_genes(mg_ranks_up, 1)[:top_n])
display(HTML(download_link(mg_ranks_up, f"multi_gene_drug_ranks_UpReg.tsv", 'Download table')))
# Down Regulation
display_markdown('**Multi-gene Drug Rank Down**', raw=True)
mg_ranks_dn = clean_ranks_df('Dn', 'mg_rank')
display(filter_genes(mg_ranks_dn, 1)[:top_n])
display(HTML(download_link(mg_ranks_dn, f"multi_gene_drug_ranks_DnReg.tsv", 'Download table')))

## deviation rank method
# Up Regulation
display_markdown('**Deviation Drug Rank Up**', raw=True)
dev_ranks_up = clean_ranks_df('Up', 'deviation')
display(filter_genes(dev_ranks_up, 1)[:top_n])
display(HTML(download_link(dev_ranks_up, f"multi_gene_drug_ranks_deviation_UpReg.tsv", 'Download table')))
# Down Regulation
display_markdown('**Deviation Drug Rank Down**', raw=True)
dev_ranks_dn = clean_ranks_df('Dn', 'deviation')
display(filter_genes(dev_ranks_dn,1)[:top_n])
display(HTML(download_link(dev_ranks_dn, f"multi_gene_drug_ranks_deviation_DnReg.tsv", 'Download table')))

# Individual gene results downloads
display_markdown("**Results per gene available for download**", raw = True)
display(HTML(download_link(clean_individual_df('Up'), f"individual_gene_drug_ranks_UpReg.tsv", 'Download table')))
display(HTML(download_link(clean_individual_df('Dn'), f"individual_gene_drug_ranks_DnReg.tsv", 'Download table')))
{% endif %}

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Single Gene' %}
## UpSet Plot

The UpSet plots show the overlap among top up-regulating or down-regulating drugs in each dataset. If there were more than 50 significant regulators in a dataset for a given input gene, the input was restricted to the top 50 regulators.
 {% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
top_up = {}
top_down = {}
# get results from Ginkgo
if in_ginkgo:
    for cell_type in ginkgo_drugs_down.keys():
        top_up[f'ginkgo_{cell_type}'] = get_top(ginkgo_drugs_up[cell_type][0], n=50)
        top_down[f'ginkgo_{cell_type}'] = get_top(ginkgo_drugs_down[cell_type][0], n=50)
# get results from novartis
if in_novartis:
    top_up['novartis'] = get_top(novartis_drugs_up[0], n=50)
    top_down['novartis'] = get_top(novartis_drugs_down[0], n=50)
# get results from Tahoe
if in_tahoe:
    top_up['tahoe'] = get_top(tahoe_drugs_up[0], n=50)
    top_down['tahoe'] = get_top(tahoe_drugs_down[0], n=50)

{% endif %}

In [ ]:
# Saving Figures
def save_figure(plot_name, **kwargs):
    import io
    mem = io.BytesIO()
    plt.savefig(mem, bbox_inches='tight')
    with open(plot_name, 'wb') as fw:
        fw.write(mem.getbuffer())

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}

def create_upset(top_sets: dict, direction: str):
    '''
    Given a dictionary to top up or down regulating genes,
    creates an Upset plot. 
    '''
    rename_keys = {
            'ginkgo_A549': 'ginkgo_A549',
            'novartis': 'novartis',
            'tahoe': 'tahoe',
            'ginkgo_human_epithelial_melanocytes': 'ginkgo_melanocytes',
            'ginkgo_human_dermal_fibroblast': 'ginkgo_fibroblasts',
            'ginkgo_human_aortic_smooth_muscle_cells': 'ginkgo_muscle_cells',
            'ginkgo_human_skeletal_muscle_myoblasts': 'ginkgo_myoblasts'
    }
    top_sets = {rename_keys[k]:v for k,v in top_sets.items()}
    upset_data = from_contents(top_sets)
    plot(upset_data, orientation = 'horizontal', show_counts = True, element_size = 30)
    save_figure(f'upset_{direction}.png')
    plt.show()

if in_ginkgo + in_novartis + in_tahoe < 2:
    display_markdown(f'**{query_gene}** not found in at least 2 datasets')
else:
    display_markdown(f"**Overlap among top up regulators of {query_gene}**", raw=True)
    create_upset(top_up, 'up')
    display(FileLink('upset_up.png', result_html_prefix='Download:'))
    display_markdown(f"**Overlap among top down regulators of {query_gene}**", raw=True)
    create_upset(top_down, 'down')
    display(FileLink('upset_down.png', result_html_prefix='Download:'))
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}

def get_overlapping_sets(top_sets:dict):
    '''
    Given the dictionary of sets used to created the UpSet plot,
    return the contents of the overlapping sets. 
    '''
    # convert to multi-index dataframe
    set_df = from_contents(top_sets)
    multi_index_df = pd.DataFrame(columns=list(top_sets.keys()))
    for colname in multi_index_df.columns:
        multi_index_df[colname] = set_df.index.get_level_values(colname).to_list()
    # only keep unique sets of intersection contributors
    multi_index_df.drop_duplicates(inplace=True)
    # sort multi-index for efficient indexing
    set_df = set_df.sort_index()
    # extract drug intersection for each group
    overlapping_sets = pd.DataFrame(columns=['Members', 'Overlap', 'Length'])
    for idx in range(multi_index_df.shape[0]):
        ixn_drugs = set_df.loc[tuple(multi_index_df.iloc[idx])].id.to_list()
        # replace PubChem IDs with drug names
        pubchem_dict = dict(zip(pubchem_ids.CID.to_list(), pubchem_ids.Drug.to_list()))
        ixn_drug_names = []
        for d in ixn_drugs:
            if d in pubchem_dict:
                ixn_drug_names.append(pubchem_dict[d])
            else:
                ixn_drug_names.append(d)
        # get group members
        ixn_name = multi_index_df.iloc[idx][multi_index_df.iloc[idx]].index.to_list()
        ixn_name_joined = '-'.join(ixn_name)
        # append results
        overlapping_sets = pd.concat([overlapping_sets, pd.DataFrame({'Members':ixn_name_joined, 'Overlap':[ixn_drug_names], 'Length':len(ixn_drug_names), 'N Datasets':len(ixn_name)})])
        
    
    overlapping_sets = overlapping_sets.sort_values('N Datasets', ascending=False)
    return overlapping_sets
 {% endif %}

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Single Gene' %}
## Consensus Regulator Tables

Below are tabular representations of the UpSet plots.
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
if in_ginkgo + in_novartis + in_tahoe < 2:
    display_markdown(f'**{query_gene}** not found in at least 2 datasets')
else:
    overlap_down = get_overlapping_sets(top_down)
    overlap_up = get_overlapping_sets(top_up)
    display_markdown("**Down-regulating drug overlap**", raw=True)
    display(overlap_down)
    display(HTML(download_link(overlap_down, f'overlapping_drugs_{query_gene}_DnReg.tsv')))
    display_markdown("**Up-regulating drug overlap**", raw=True)
    display(overlap_up)
    display(HTML(download_link(overlap_up, f'overlapping_drugs_{query_gene}_UpReg.tsv')))
{% endif %}


In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
def get_ranking_averages(overlapping_df, data_dict, ranking_method):
    # get average, integrating across datasets
    average_rank_vals = {}
    average_pctrank_vals = {}
    average_logfc_vals = {}
    average_pvals = {}
    n_datasets = list()
    pubchem_dict = dict(zip(pubchem_ids.Drug.to_list(), pubchem_ids.CID.to_list()))
    pubchem_dict_rev = dict(zip(pubchem_ids.CID.to_list(), pubchem_ids.Drug.to_list()))
    for _,row in overlapping_df.iterrows():
        n_datasets.extend([row['N Datasets']]*len(row['Overlap']))
        member_sets = row['Members']
        for d in row['Overlap']:
            n = 0
            runsum_rank = 0
            runsum_pctrank = 0
            runsum_logFC = 0
            runsum_pval = 0
            for source_name,df in data_dict.items():
                if not re.search(source_name, member_sets):
                    continue
                d = pubchem_dict[d] if d in pubchem_dict else d
                subset = df[df['DrugGroup'] == d]
                n = n + subset.shape[0]
                runsum_rank = runsum_rank + subset.Rank.sum()
                runsum_pctrank = runsum_pctrank + subset.PctRank.sum()
                runsum_logFC = runsum_logFC + subset.logFC.sum()
                runsum_pval = runsum_pval + subset['adj.P.Val'].sum()
            average_rank_vals[d] = round(runsum_rank / n,3)
            average_pctrank_vals[d] = round(runsum_pctrank/n, 3)
            average_logfc_vals[d] = round(runsum_logFC/n, 3)
            average_pvals[d] = round(runsum_pval/n,3)
    # create results dataframe
    res_df = pd.DataFrame({
        'Drug': list(average_rank_vals.keys()),
        'Avg LogFC': list(average_logfc_vals.values()),
        'Avg Adj.P.Val': list(average_pvals.values()),
        'Avg Rank': list(average_rank_vals.values()),
        'Avg PctRank': list(average_pctrank_vals.values())
    })
    res_df['N Datasets'] = n_datasets
    res_df['Drug'] = res_df['Drug'].apply(lambda x: pubchem_dict_rev[x] if x in pubchem_dict_rev else x)
    if ranking_method == 'target_rank':
        # sort based on N datasets and average percentile rank
        res_df = res_df.sort_values(['N Datasets','Avg PctRank'], ascending=[False,True])
    else:
        # sort based on N datasets and average adjusted p-value
        res_df = res_df.sort_values(['N Datasets','Avg Adj.P.Val'], ascending=[False,True])
    return res_df


def join_proteomics(ranking_table, protein_de):
    # join with PubChem ID table
    with_cids = ranking_table.merge(pubchem_ids, how='left', on='Drug')
    # Drop those drugs that did not have PubChem IDs
    with_cids = with_cids[with_cids['CID'].notna()]
    # join with proteomics data on PubChem IDs
    with_proteins = with_cids.merge(protein_de[['UniprotID','Pubchem','logFC']], how='inner', left_on='CID', right_on='Pubchem')
    # clean column names
    with_proteins.rename(columns = {'logFC':'Protein logFC', 'Pubchem' : 'PubChem CID'}, inplace=True)
    with_proteins.drop(columns='CID',inplace=True)
    return with_proteins.sort_values(['N Datasets', 'Avg Adj.P.Val'], ascending=[False,True])
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
# define input data for get_ranking_averages
data_source_present = {'A549': in_ginkgo, 
                    'human_dermal_fibroblast':in_ginkgo,
                    'human_aortic_smooth_muscle_cells':in_ginkgo,
                    'human_epithelial_melanocytes': in_ginkgo,
                    'human_skeletal_muscle_myoblasts': in_ginkgo,
                    'novartis': in_novartis,
                    'tahoe': in_tahoe}
data_dict_down ={}
data_dict_up = {}
for source,present in data_source_present.items():
    if (present) & (not source in ['novartis','tahoe']):
        data_dict_down[source] = ginkgo_drugs_down[source][1]
        data_dict_up[source] = ginkgo_drugs_up[source][1]
    elif (present) & (source == 'novartis'):
        data_dict_down[source] = novartis_drugs_down[1]
        data_dict_up[source] = novartis_drugs_up[1]
    elif (present) & (source == 'tahoe'):
        data_dict_down[source] = tahoe_drugs_down[1]
        data_dict_up[source] = tahoe_drugs_up[1]

if in_ginkgo + in_novartis + in_tahoe > 1:
    overlapping_up_TargetRank = get_ranking_averages(overlap_up, data_dict_up, ranking_method)
    overlapping_down_TargetRank = get_ranking_averages(overlap_down, data_dict_down, ranking_method)
{% endif %}

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Single Gene' %}
The tables below show average logFC, adjusted p-value, raw rank and normalized rank values across datasets for drugs that were found to be significant regulators in more than one dataset.
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
if in_ginkgo + in_novartis + in_tahoe < 2:
    display_markdown(f'**{query_gene}** not found in at least 2 datasets')
else:
    display_markdown("**Averages across datasets: Up-regulating drugs**", raw=True)
    display(overlapping_up_TargetRank.head(n=top_n))
    display(HTML(download_link(overlapping_up_TargetRank, f'overlapping_drugs_averages_{query_gene}_UpReg.tsv')))
    display_markdown("**Averages across datasets: Down-regulating drugs**", raw=True)
    display(overlapping_down_TargetRank.head(n=top_n))
    display(HTML(download_link(overlapping_down_TargetRank, f'overlapping_drugs_averages_{query_gene}_DnReg.tsv')))
{% endif %}

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Single Gene' %}
### Protein Regulation

Query gene regulation at the protein level is dispalyed in the table below. Proteomics data is from the [Deepcover MoA dataset](https://wren.hms.harvard.edu/DeepCoverMOA/), which exposes cells from the HCT116 cancer cell line to 875 small molecule compounds. 
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
if in_deepcover:
    up_protein = protein_de[protein_de['logFC'] > 0].loc[:,['UniprotID','Drug','Pubchem','logFC','Zscore','UpRank','PctUpRank']].sort_values('logFC',ascending=False).reset_index().drop(columns='index')
    up_protein.rename(columns={'Pubchem':'PubChem CID','Zscore':'Z-score','UpRank':'Up Rank', 'PctUpRank':'Normalized Up Rank'}, inplace=True)
    display_markdown("**Up-regulating drugs**", raw=True)
    display(up_protein.head(top_n))
    display(HTML(download_link(up_protein, f'DeepcoverMoa_protein_{query_gene}_UpReg.tsv')))

    dn_protein = protein_de[protein_de['logFC'] < 0].loc[:,['UniprotID','Drug','Pubchem','logFC','Zscore','DnRank','PctDnRank']].sort_values('logFC', ascending=True).reset_index().drop(columns='index')
    dn_protein.rename(columns={'Pubchem':'PubChem CID','Zscore':'Z-score','UpRDnRankank':'Down Rank', 'PctDnRank':'Normalized Down Rank'}, inplace=True)
    display_markdown("**Down-regulating drugs**", raw=True)
    display(dn_protein.head(top_n))
    display(HTML(download_link(dn_protein, f'DeepcoverMoa_protein_{query_gene}_DnReg.tsv')))
else:
    display_markdown(f"Protein of {query_gene} not in DeepCover MoA Dataset", raw=True)
{% endif %}

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Single Gene' %}
If the protein associated with the query gene was found in the Deepcover MoA proteomics dataset, the tables below show how the protein was up or down-regulated by the consensus drugs identified in the connectivity mapping resources. The table only includes compounds that were used in the connectivity mapping resources and the Deepcover MoA dataset. 
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
if in_deepcover:
    up_with_cid = join_proteomics(overlapping_up_TargetRank, protein_de)
    dn_with_cid = join_proteomics(overlapping_down_TargetRank, protein_de)
    
    display_markdown("**Up-regulating drugs with protein expression**", raw=True)
    display(up_with_cid.head(n=top_n))
    display(HTML(download_link(up_with_cid, f'{query_gene}_mRNA_protein_UpReg.tsv')))
    
    display_markdown("**Down-regulating drugs with protein expression**", raw=True)
    display(dn_with_cid.head(n=top_n))
    display(HTML(download_link(dn_with_cid, f'{query_gene}_mRNA_protein_DnReg.tsv')))
else:
    display_markdown(f"Protein of {query_gene} not in DeepCover MoA Dataset", raw=True)

{% endif %}

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Single Gene' %}
## Venn Diagrams

The venn diagrams show the pairwise overlap among either up-regulating or down-regulating drugs across the four Connectivity Mapping datasets Tahoe-100M, Novartis DRUG-seq, and Ginkgo (all cell types grouped). 

{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
# combine top up and down drugs across Ginkgo cell types
if in_ginkgo:
    all_ginkgo_up = set()
    all_ginkgo_down = set()
    for k,v in top_up.items():
        if re.search('ginkgo',k):
            all_ginkgo_up = all_ginkgo_up.union(v)
            all_ginkgo_down = all_ginkgo_down.union(top_down[k])
# define input data for venn diagrams
data_source_present = {'ginkgo':in_ginkgo,
                    'novartis':in_novartis,
                    'tahoe': in_tahoe}
venn_up = {}
venn_down = {}
for source,present in data_source_present.items():
    if (present) & (source == 'ginkgo'):
        venn_up[source] = all_ginkgo_up
        venn_down[source] = all_ginkgo_down
    elif present:
        venn_up[source] = top_up[source]
        venn_down[source] = top_down[source]

{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}

def create_venn(venn_dict):
    '''
    Given a dictionary of sets of top up or down regulating genes,
    creates a 2 or 3 set venn diagram. 
    '''
    if len(venn_dict) == 3:
        venn3(subsets=(venn_dict['novartis'], venn_dict['tahoe'], venn_dict['ginkgo']),
            set_labels=('Novartis', 'Tahoe', 'Ginkgo'));
    elif len(venn_dict) == 2:
        venn2(subsets = (venn_dict.values()),
              set_labels=(venn_dict.keys()));


def print_overlap(venn_dict):
    '''
    Given a dictionary of sets of top up or down regulating genes,
    prints the unique overlap among all combinations of sets (>= 2). 
    '''
    results = {}
    datasets = list(venn_dict.keys())
    # Step 1: get all intersections
    for r in range(2, len(datasets)+1):
        for combo in combinations(datasets, r):
            drug_sets = [venn_dict[d] for d in combo]
            results[combo] = list(set.intersection(*drug_sets))
    # Step 2: subtract intersections of all super sets 
    for combo, base_set in results.items():
        unique_set = set(base_set)
        # iterate through all sets from larger combinations
        for r in range(len(combo) + 1, len(datasets) + 1):
            for sup in combinations(datasets, r):
                if set(combo).issubset(sup):  # only consider supersets
                    super_set = set(results[sup])
                    # get difference between super set and current set, this
                    # is the new intersection for the combination
                    new_unique_set = unique_set - super_set
                    results[combo] = list(new_unique_set)
    # Step 3: print results
    for datasets, overlap in results.items():
        if len(overlap) == 0:
            overlap = ['None']
        # print(f"{', '.join(datasets)}: {', '.join(overlap)}")
        return f"{', '.join(datasets)}: {', '.join(overlap)}"
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
if in_ginkgo + in_novartis + in_tahoe < 2:
    display_markdown(f'**{query_gene}** not found in at least 2 datasets')
else:
    display_markdown(f'Overlap of top {query_gene} up-regulating drugs across sources', raw=True)
    for combo in combinations(list(venn_up.keys()), 2):
        combo_venn = {k:venn_up[k] for k in combo if k in venn_up}
        create_venn(combo_venn)
        title = print_overlap(combo_venn)
        plt.title(title)
        plt_name = f"{title.split(':')[0].replace(',','_')}_up.png"
        save_figure(plt_name)
        plt.show()
        display(FileLink(plt_name, result_html_prefix='Download: '))
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
if in_ginkgo + in_novartis + in_tahoe < 2:
    display_markdown(f'**{query_gene}** not found in at least 2 datasets')
else:
    display_markdown(f'Overlap of top {query_gene} down-regulating drugs across sources', raw=True)
    for combo in combinations(list(venn_down.keys()), 2):
        combo_venn = {k:venn_down[k] for k in combo if k in venn_down}
        create_venn(combo_venn)
        title = print_overlap(combo_venn)
        plt.title(title)
        plt_name = f"{title.split(':')[0].replace(',','_')}_down.png"
        save_figure(plt_name)
        plt.show()
        display(FileLink(plt_name, result_html_prefix='Download: '))
{% endif %}

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Multi-gene' %}
## Top 20 Drug Plots

The barplots show the multi-gene drug rank for the top 20 drugs for up and down regulation of the input geneset. The top drugs are identified by two sorting methods.

1. Multi-gene rank: the average ranking for the drug acorss the genes in the input gene set

2. Deviation rank: difference between multi-gene rank and the expected rank of the drug, determined by retrieving drug ranks for random input gene sets. 


Plots are shown with and without filtering for drugs that regulate a majority of the inpute gene set in the specified direction. 

**Drug annotations (Y-axis):**
* \*\* FDA approved drug
* <span style="color: red;">Red</span>: Universal regulator, determined based on expected drug rank calculated from random input gene sets. 

{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Multi-gene' %}
def visualize_top_drugs(direction, max_genes=10, df=None, rank_method='MultiGeneRank', flag_filter = False, filter_genes=False):
    from math import ceil 

    top_n_drugs = 20
             
    # Create a blue-white-red continuous colormap centered at max_genes / 2
    cmap = sns.color_palette("coolwarm", as_cmap=True)
    norm = TwoSlopeNorm(vmin=1, vcenter=max_genes / 2, vmax=max_genes)

    # join with expected rank data
    if direction == 'Down':
        df = df.merge(null_ranks_dn[['DrugMatch','NullRank']], how='left', on='DrugMatch')
    elif direction == 'Up':
        df = df.merge(null_ranks_up[['DrugMatch','NullRank']], how='left', on='DrugMatch')
    
    df['RankDiff'] = df['NullRank']-df['MultiGeneRank']
    
    # remove flagged drugs
    if flag_filter:
        df = df[~df['DrugMatch'].isin(flagged_drugs['DrugMatch'].to_list())]
    if filter_genes:
        df = df[df['NGenes'] >= ceil(max_genes/2)]

    # select top n drugs by RankDiff
    if rank_method == 'RankDiff':
        method_title = 'Deviation Rank'
        df = df.sort_values(['RankDiff','MultiGeneRank', 'NGenes'], ascending=[False,True,False])[:top_n_drugs]
    # select top n drugs by MG Rank
    elif rank_method == 'MultiGeneRank':
        method_title = 'Multi-gene Rank'
        df = df.sort_values(['MultiGeneRank', 'NGenes'], ascending=[True,False])[:top_n_drugs]
    
    # FDA status
    fda_approved = df.merge(fda_approved_toxicity, how='left', left_on='DrugMatch',right_on='synonyms_lower')
    fda_approved = list(fda_approved[fda_approved['isApproved']==True].DrugMatch.unique())
    
    # order for plotting
    df = df.sort_values('MultiGeneRank', ascending=True)
    fig, ax = plt.subplots(figsize = (6,8))
    null_bar = sns.barplot(df,
                    y='DrugMatch',
                    x='NullRank',
                    color='darkgray',
                    ax=ax)
    mg_bar = sns.barplot(df,
                y='DrugMatch',
                x='MultiGeneRank',
                hue='NGenes',
                palette=cmap,
                hue_norm=norm,
                ax=ax)
   
    plt.ylabel(None)
    plt.xlabel('Multi-gene Drug Rank [0-1]')
    plt.title(f'{method_title} Top {top_n_drugs} Drugs: {direction}')
    # add legend
    top_bar = mpatches.Patch(color='darkgray', label='Null Rank')
    plt.legend(handles=[top_bar], loc='upper center', bbox_to_anchor=(0.5,-0.1))
    # add colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([]) # needed for fig.colorbar
    cbar = fig.colorbar(sm, ax=ax, orientation='horizontal', fraction=0.03, location='bottom')
    cbar.set_label(f'Number of input targets {direction}-regulated')
    
    # Update specified y-axis labels
    star_set = set([str(x) for x in fda_approved])

    # Denote FDA approved drugs
    modified_label = []
    for label in ax.get_yticklabels():
        txt = label.get_text()
        if txt in star_set:
            modified_label.append(f'**{txt}')
        else:
            modified_label.append(txt)
    
    # Redraw the modified labels
    ax.set_yticklabels(modified_label)

    # Denotes flagged drugs
    for label in ax.get_yticklabels():
        txt = label.get_text()
        if str.replace(txt,'**','') in flagged_drugs['DrugMatch'].to_list():
            label.set_color('red')
        else:
            label.set_color('black')
    gene_filter = 'majorityGenes' if filter_genes else ''       
    plt_name = f"{direction}_{rank_method}_top{top_n_drugs}_{gene_filter}.png"
    save_figure(plt_name)
    plt.show()
    return plt_name
    
# Up
display_markdown(" ### Up-Regulating", raw=True)
display_markdown(" #### Multi-gene Rank", raw=True)
display(HTML('<p style="text-align:center; font-weight:bold";>Filter for drugs that regulate at least half of the input genes</p>'))
file_name = visualize_top_drugs('Up', max_genes=len(query_geneset), df = multi_gene_drug_ranks['Up'], rank_method='MultiGeneRank', filter_genes=True)
display(FileLink(file_name, result_html_prefix='Download: '))
display(HTML('<p style="text-align:center; font-weight:bold";>No gene regulation filter</p>'))
file_name = visualize_top_drugs('Up', max_genes=len(query_geneset), df = multi_gene_drug_ranks['Up'], rank_method='MultiGeneRank', filter_genes=False)
display(FileLink(file_name, result_html_prefix='Download: '))

display_markdown(" #### Deviation Rank", raw=True)
display(HTML('<p style="text-align:center; font-weight:bold";>Filter for drugs that regulate at least half of the input genes</p>'))
file_name = visualize_top_drugs('Up', max_genes=len(query_geneset), df = multi_gene_drug_ranks['Up'], rank_method='RankDiff', filter_genes=True)
display(FileLink(file_name, result_html_prefix='Download: '))
display(HTML('<p style="text-align:center; font-weight:bold";>No gene regulation filter</p>'))
file_name = visualize_top_drugs('Up', max_genes=len(query_geneset), df = multi_gene_drug_ranks['Up'], rank_method='RankDiff', filter_genes=False)
display(FileLink(file_name, result_html_prefix='Download: '))

# Dn
display_markdown(" ### Down-Regulating", raw=True)
display_markdown(" #### Multi-gene Rank", raw=True)
display(HTML('<p style="text-align:center; font-weight:bold";>Filter for drugs that regulate at least half of the input genes</p>'))
file_name=visualize_top_drugs('Down', max_genes=len(query_geneset), df = multi_gene_drug_ranks['Dn'], rank_method='MultiGeneRank', filter_genes=True)
display(FileLink(file_name, result_html_prefix='Download: '))
display(HTML('<p style="text-align:center; font-weight:bold";>No gene regulation filter</p>'))
file_name=visualize_top_drugs('Down', max_genes=len(query_geneset), df = multi_gene_drug_ranks['Dn'], rank_method='MultiGeneRank', filter_genes=False)
display(FileLink(file_name, result_html_prefix='Download: '))

display_markdown(" #### Deviation Rank", raw=True)
display(HTML('<p style="text-align:center; font-weight:bold";>Filter for drugs that regulate at least half of the input genes</p>'))
file_name=visualize_top_drugs('Down', max_genes=len(query_geneset), df = multi_gene_drug_ranks['Dn'], rank_method='RankDiff', filter_genes=True)
display(FileLink(file_name, result_html_prefix='Download: '))
display(HTML('<p style="text-align:center; font-weight:bold";>No gene regulation filter</p>'))
file_name=visualize_top_drugs('Down', max_genes=len(query_geneset), df = multi_gene_drug_ranks['Dn'], rank_method='RankDiff', filter_genes=False)
display(FileLink(file_name, result_html_prefix='Download: '))

{% endif %}

## Volcano Plots

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Single Gene' %}
The volcano plots show the strength and statistical significance of the drug perturbation for each signature in the dataset (drug and dose specific). Color of points indicate up (red) or down (blue) regulation. Hover over points in the volcano plot to see the label (with cell line, drug, and dose information), logFC, fold-change, log10-transformed p-value, raw rank and normalized rank. Tools to the right of the plot allow you to manipulate (pan, zoom) and download the figure. 

{% elif gene_input.raw_value == 'Multi-gene' %}
The volcano plots show the strength and statistical significance of the drug perturbation for each signature in the datasets. Color of points indicate the dataset associated with the signature. Hover over points in the volcano plot to see the perturbation information, logFC, fold-change, log10-transformed p-value, raw rank and normalized rank. Tools to the right of the plot allow you to manipulate (pan, zoom) and download the figure. 
{% endif %}

In [ ]:
output_notebook()

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
def create_bokeh_volcano_plot(expr_data:pd.DataFrame, gene_id:str, cell_type:str, source:str):
    '''
    Given the expression data for a given gene, create an interactive
    volcano plot that shows regulation of gene across all perturbations (drug, dosage, cell line).
    '''
    
    df = expr_data.copy()
    
    # clean columns
    df['FC'] = 2**df['logFC']
    if source == 'Ginkgo':
        df['Label'] = df['Perturbation']
    elif source == 'Novartis':
        df['Label'] = df['Perturbation'] + '_' + df['Drug']
    elif source == 'L1000':
        df['Label'] = df['Perturbation']
    elif source == 'Tahoe':
        df['Label'] = df['Drug'] + '-' + df['concentration'].astype(str) + '-' + df['Cell_ID_Cellosaur']
    elif source == 'Deepcover MoA':
        df['Label'] = df['Drug']
        df['abs_Zscore'] = df['Zscore'].apply(abs)

    # set plot source
    if source != 'Deepcover MoA':
        plot_source = ColumnDataSource(df.loc[:,['Label','logFC','FC','log10adj.P.Val', 'Rank', 'PctRank']])
        x,y='logFC','log10adj.P.Val'
        xlabel,ylabel = 'Log2(Fold Change)','-Log10(Adj. p-value)'
        title = f'{gene_id} Regulation in {source} {cell_type}'
        hover = HoverTool(tooltips=[("Label", "@Label"),
                            ("Log2(FC)", "@logFC"),
                            ("Fold Change", "@FC"),
                            ('-Log10(Adj. p-value)',"@{log10adj.P.Val}{0.00e}"),
                            ("Raw Rank", "@Rank"),
                            ("Normalized Rank", "@PctRank")])
    else:
        plot_source = ColumnDataSource(df.loc[:,['Label','logFC','FC','abs_Zscore','UpRank','DnRank','PctUpRank','PctDnRank']])
        x,y = 'logFC','abs_Zscore'
        xlabel,ylabel = 'Log2(Fold Change)','Abs(Z-score)'
        title = f'{gene_id}: {df["UniprotID"].iloc[0]} Regulation in {source} {cell_type}'
        hover = HoverTool(tooltips=[("Label", "@Label"),
                            ("Log2(FC)", "@logFC"),
                            ("Fold Change", "@FC"),
                            ('abs(z-score)',"@{abs_Zscore}{0.00e}"),
                            ("Up Rank", "@UpRank"),
                            ("Normalized Up Rank","@PctUpRank"),
                            ("Down Rank", "@DnRank"),
                            ("Normalized Down Rank","@PctDnRank")])

    
        
    # define figure
    p = figure(
        title=title,
        x_axis_label = xlabel,
        y_axis_label = ylabel,
        tools = 'pan,wheel_zoom,box_zoom,reset,save'
    )

    # color mapper
    color_mapper = LinearColorMapper(palette = RdBu[10],
                                     low = min(df['logFC']),
                                     high=max(df['logFC']))
    # plot
    p.scatter(x=x,
              y=y,
              size=8,
              source=plot_source,
              fill_alpha=0.6,
              color = {'field':'logFC','transform':color_mapper})
    p.add_tools(hover)
    show(p)
    
{% endif %}

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Single Gene' %}
### Ginkgo
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
if in_ginkgo:
    for cell_type, expr_df in ginkgo_gene_expr_dict.items():
        cell_name = ' '.join(re.sub('human_','',cell_type).split('_'))
        create_bokeh_volcano_plot(expr_df, query_gene, cell_name, 'Ginkgo')       
else:
    display_markdown(f'**{query_gene}** not found in Ginkgo datasets', raw=True)
{% endif %}

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Single Gene' %}
### Novartis DRUG-seq
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
if in_novartis:
    create_bokeh_volcano_plot(novartis_de, query_gene, '', 'Novartis')   
else:
    display_markdown(f'**{query_gene}** not found in Novartis DRUG-seq dataset', raw=True)
{% endif %}

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Single Gene' %}
### Tahoe-100M
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}

if in_tahoe:
    create_bokeh_volcano_plot(tahoe_de, query_gene, '','Tahoe')
else:
    display_markdown(f'**{query_gene}** not found in Tahoe-100M dataset', raw=True)

{% endif %}

In [ ]:
%%appyter markdown
{% if gene_input.raw_value == 'Single Gene' %}
### Deepcover MoA

The Deepcover MoA proteomics dataset consists of proteome fingerprints for 875 chemical perturbations. This volcano plot of protein expression shows the logFC on the x-axis and the absolute difference in standard deviations between the protein's logFC for a given compound and the protein's mean logFC across all compounds on the y-axis. 
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Single Gene' %}
if in_deepcover:
    # check for multiple proteins
    uniprot_ids = list(protein_de['UniprotID'].unique())
    for uid in uniprot_ids:
        create_bokeh_volcano_plot(protein_de[protein_de['UniprotID']==uid], query_gene, '', 'Deepcover MoA')
{% endif %}

In [ ]:
%%appyter code_exec
{% if gene_input.raw_value == 'Multi-gene' %}

def create_bokeh_volcano_plot(expr_data:pd.DataFrame, gene_id:str):
    '''
    Given the expression data for a given gene, create an interactive
    volcano plot that shows regulation of gene across all perturbations (drug, dosage, cell line).
    '''
    
    df = expr_data.copy()
    
    # clean columns
    df['FC'] = 2**df['logFC']

    # set plot source
  
    plot_source = ColumnDataSource(df.loc[:,['Perturbation','Drug','Dataset','logFC','FC','log10adj.P.Val','PctRank']])
    x,y='logFC','log10adj.P.Val'
    xlabel,ylabel = 'Log2(Fold Change)','-Log10(Adj. p-value)'
    title = f'{gene_id} Regulation'
    hover = HoverTool(tooltips=[("Perturbation", "@Perturbation"),
                        ("Drug", "@Drug"),
                        ("Dataset", "@Dataset"),
                        ("Log2(FC)", "@logFC"),
                        ("Fold Change", "@FC"),
                        ('-Log10(Adj. p-value)',"@{log10adj.P.Val}{0.00e}"),
                        ("Normalized Rank", "@PctRank")])

    
        
    # define figure
    p = figure(
        title=title,
        x_axis_label = xlabel,
        y_axis_label = ylabel,
        tools = 'pan,wheel_zoom,box_zoom,reset,save'
    )

    # color mapper
    color_mapper = CategoricalColorMapper(palette = Category10[3],
                                     factors = ['ginkgo', 'tahoe', 'novartis'])
    # plot
    p.scatter(x=x,
              y=y,
              size=8,
              source=plot_source,
              fill_alpha=0.6,
              color = {'field':'Dataset', 'transform': color_mapper})
    p.add_tools(hover)
    show(p)

for g in query_geneset:
    if g in all_genes:
        create_bokeh_volcano_plot(all_genes[g], g)
{% endif %}

## References

[1] Baugh, Lauren, Sébastien Vigneau, Srijani Sridhar, Sarah Boswell, George Pilitsis, John Bradley, Olga Allen, et al. 2025. “Mapping the Transcriptional Landscape of Drug Responses in Primary Human Cells Using High-Throughput DRUG-Seq.” bioRxiv. https://doi.org/10.1101/2025.06.03.657593.

[2] Datapoints, Ginkgo. n.d. “GDPx1.” Accessed September 5, 2025. https://huggingface.co/datasets/ginkgo-datapoints/GDPx1.

[3] Hadjikyriacou, Andrea, Chian Yang, Martin Henault, Robin Ge, Leandra Mansur, Alicia Lindeman, Carsten Russ, et al. 2025. “Novartis/DRUG-Seq U2OS MoABox Dataset.” Zenodo. https://doi.org/10.5281/ZENODO.14291446.

[4] Subramanian, Aravind, Rajiv Narayan, Steven M. Corsello, David D. Peck, Ted E. Natoli, Xiaodong Lu, Joshua Gould, et al. 2017. “A next Generation Connectivity Map: L1000 Platform and the First 1,000,000 Profiles.” Cell 171 (6): 1437-1452.e17.

[5] Mitchell, Dylan C., Miljan Kuljanin, Jiaming Li, Jonathan G. Van Vranken, Nathan Bulloch, Devin K. Schweppe, Edward L. Huttlin, and Steven P. Gygi. 2023. “A Proteome-Wide Atlas of Drug Mechanism of Action.” Nature Biotechnology 41 (6): 845–57.

[6] Zhang, Jesse, Airol A. Ubas, Richard de Borja, Valentine Svensson, Nicole Thomas, Neha Thakar, Aidan Winters, et al. 2025. “Tahoe-100M: A Giga-Scale Single-Cell Perturbation Atlas for Context-Dependent Gene Function and Cellular Modeling.” bioRxiv. https://doi.org/10.1101/2025.02.20.639398.

[7] Wang, Zichen, Edward He, Kevin Sani, Kathleen M. Jagodnik, Moshe C. Silverstein, and Avi Ma’ayan. 2019. “Drug Gene Budger (DGB): An Application for Ranking Drugs to Modulate a Specific Gene Based on Transcriptomic Signatures.” Bioinformatics (Oxford, England) 35 (7): 1247–48.